In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel('data.xlsx')
df.head()

,ID,WBC,LYMp,MIDp,NEUTp,LYMn,MIDn,NEUTn,RBC,HGB,...,MCV,MCH,MCHC,RDWSD,RDWCV,PLT,MPV,PDW,PCT,PLCR
0,1,10.0,43.2,6.7,50.1,4.3,0.7,5.0,2.77,7.3,...,87.7,26.3,30.1,35.3,11.4,189.0,9.2,12.5,0.17,22.3
1,2,10.0,42.4,5.3,52.3,4.2,0.5,5.3,2.84,7.3,...,88.2,25.7,20.2,35.3,11.4,180.0,8.9,12.5,0.16,19.5
2,3,7.2,30.7,8.6,60.7,2.2,0.6,4.4,3.97,9.0,...,77.0,22.6,29.5,37.2,13.7,148.0,10.1,14.3,0.14,30.5
3,4,6.0,30.2,6.3,63.5,1.8,0.4,3.8,4.22,3.8,...,77.9,23.2,29.8,46.5,17.0,143.0,8.6,11.3,0.12,16.4
4,5,4.2,39.1,7.2,53.7,1.6,0.3,2.3,3.93,0.4,...,80.6,23.9,29.7,42.7,15.1,236.0,19.5,12.8,0.22,24.8


In [3]:
df=df[['MCV','MCH','MCHC','RDWCV','HGB']]
df.head()

,MCV,MCH,MCHC,RDWCV,HGB
0,87.7,26.3,30.1,11.4,7.3
1,88.2,25.7,20.2,11.4,7.3
2,77.0,22.6,29.5,13.7,9.0
3,77.9,23.2,29.8,17.0,3.8
4,80.6,23.9,29.7,15.1,0.4


In [4]:
df.isnull().sum()

MCV      0
MCH      0
MCHC     0
RDWCV    0
HGB      0
dtype: int64

In [5]:
(df<0).sum()

MCV      1
MCH      0
MCHC     0
RDWCV    0
HGB      1
dtype: int64

In [6]:
df=df[(df>=0).all(axis=1)]

In [7]:
(df<0).sum()

MCV      0
MCH      0
MCHC     0
RDWCV    0
HGB      0
dtype: int64

In [8]:
df

,MCV,MCH,MCHC,RDWCV,HGB
0,87.7,26.3,30.1,11.4,7.3
1,88.2,25.7,20.2,11.4,7.3
2,77.0,22.6,29.5,13.7,9.0
3,77.9,23.2,29.8,17.0,3.8
4,80.6,23.9,29.7,15.1,0.4
...,...,...,...,...,...
495,86.4,27.4,31.7,12.2,13.2
496,76.7,24.0,31.4,13.8,11.6
497,68.8,22.7,33.0,10.6,9.9
498,70.6,21.9,30.9,11.0,7.4


In [12]:
MCV_ref=90
MCH_ref=30
MCHC_ref=34
RDW_ref=13
HGB_ref=14

In [13]:
D_MCV = np.maximum((MCV_ref - df['MCV']) / MCV_ref, 0)
D_MCH = np.maximum((MCH_ref - df['MCH']) / MCH_ref, 0)
D_MCHC = np.maximum((MCHC_ref - df['MCHC']) / MCHC_ref, 0)
D_RDW = np.maximum((df['RDWCV'] - RDW_ref) / RDW_ref, 0)
D_HGB = np.maximum((HGB_ref - df['HGB']) / HGB_ref, 0)

In [14]:
df["Severity"] = (
    0.25 * D_MCV +
    0.20 * D_MCH +
    0.20 * D_MCHC +
    0.15 * D_RDW +
    0.20 * D_HGB
)

In [15]:
df.head()

,MCV,MCH,MCHC,RDWCV,HGB,Severity
0,87.7,26.3,30.1,11.4,7.3,0.149711
1,88.2,25.7,20.2,11.4,7.3,0.210557
2,77.0,22.6,29.5,13.7,9.0,0.191421
3,77.9,23.2,29.8,17.0,3.8,0.295518
4,80.6,23.9,29.7,15.1,0.4,0.310588


In [17]:
df['Severity'].describe()

count    498.000000
mean       0.131247
std        0.107444
min        0.000000
25%        0.062366
50%        0.108478
75%        0.173168
max        1.346344
Name: Severity, dtype: float64

In [9]:
from sklearn.model_selection import train_test_split

In [18]:
X=df[['MCV','MCH','MCHC','RDWCV','HGB']]
Y=df['Severity']
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [20]:
X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((398, 5), (100, 5), (398,), (100,))

In [22]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(X_train)
X_train_scaled=scaler.transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [23]:
X_train_scaled=pd.DataFrame(X_train_scaled,columns=X_train.columns)
X_test_scaled=pd.DataFrame(X_test_scaled,columns=X_test.columns)

In [24]:
np.round(X_train_scaled.describe(),1)

,MCV,MCH,MCHC,RDWCV,HGB
count,398.0,398.0,398.0,398.0,398.0
mean,0.0,-0.0,-0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0
min,-6.0,-0.1,-4.9,-0.7,-1.9
25%,-0.4,-0.1,-0.3,-0.2,-0.3
50%,0.2,-0.1,0.0,-0.1,-0.1
75%,0.6,-0.1,0.3,0.0,0.2
max,2.7,15.5,12.2,17.3,12.4
